# Task 5: Auto Tagging Support Tickets Using LLM

## Problem Statement & Objective
Automatically categorize support tickets into predefined tags using Large Language Models. This task compares zero-shot, few-shot, and fine-tuned approaches to understand their performance differences.

**Categories:** Billing, Technical, Account, General

## 1. Import Required Libraries

In [1]:
from langchain.llms import AzureOpenAI
from langchain.chat_models import AzureChatOpenAI
from langchain.prompts import PromptTemplate
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import json
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# Load environment variables from .env
load_dotenv('/media/arham/90E2383BE238283C/Data/Programs/potti/.env')

print("All libraries imported successfully!")
print("✓ Azure OpenAI (GPT-4.1) configured from .env")

All libraries imported successfully!
✓ Azure OpenAI (GPT-4.1) configured from .env


## 2. Dataset Loading & Preprocessing

In [2]:
# Create synthetic support ticket dataset
np.random.seed(42)

support_tickets = [
    # Billing tickets
    "I was charged twice for my subscription this month. Can you refund the duplicate charge?",
    "Why is my invoice amount higher than expected? Please explain the charges.",
    "I need an invoice for my recent purchase. The amount seems incorrect.",
    "Can I change my payment plan? I want to downgrade to a cheaper option.",
    "When will I receive my refund? I requested it 2 weeks ago.",
    
    # Technical tickets
    "The app keeps crashing when I try to upload files. Error code 500.",
    "I'm getting a connection timeout error. Is the server down?",
    "The API integration is not working with our system. Can you help debug?",
    "The website is loading very slowly. Can you optimize performance?",
    "I cannot download my data export. The file seems corrupted.",
    
    # Account tickets
    "I forgot my password. How can I reset it?",
    "I want to delete my account permanently. What's the process?",
    "How do I update my profile information and contact details?",
    "I need to link a second email address to my account.",
    "My account is locked. How do I unlock it?",
    
    # General tickets
    "What are your business hours? When can I call support?",
    "Do you have a mobile app? Where can I download it?",
    "What payment methods do you accept?",
    "Can I upgrade my plan without losing my current data?",
    "How do I contact your sales team for a custom solution?",
] * 5  # Repeat 5 times for more samples

# Create labels
labels = (
    ["Billing"] * 5 +
    ["Technical"] * 5 +
    ["Account"] * 5 +
    ["General"] * 5
) * 5

# Create DataFrame
df = pd.DataFrame({
    'ticket_text': support_tickets,
    'category': labels
})

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset created with {len(df)} support tickets")
print(f"\nCategory distribution:")
print(df['category'].value_counts())
print(f"\nSample tickets:")
for i in range(3):
    print(f"\n{i+1}. {df.iloc[i]['ticket_text'][:80]}...")
    print(f"   Category: {df.iloc[i]['category']}")

Dataset created with 100 support tickets

Category distribution:
category
Billing      25
Account      25
Technical    25
General      25
Name: count, dtype: int64

Sample tickets:

1. Can I change my payment plan? I want to downgrade to a cheaper option....
   Category: Billing

2. I need to link a second email address to my account....
   Category: Account

3. I forgot my password. How can I reset it?...
   Category: Account


In [3]:
# Split data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['ticket_text'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# Select few-shot examples
few_shot_examples = [
    ("I was charged twice for my subscription this month. Can you refund the duplicate charge?", "Billing"),
    ("The app keeps crashing when I try to upload files. Error code 500.", "Technical"),
    ("I forgot my password. How can I reset it?", "Account"),
    ("What are your business hours? When can I call support?", "General"),
]

print(f"Train set: {len(X_train)} tickets")
print(f"Test set: {len(X_test)} tickets")
print(f"Few-shot examples: {len(few_shot_examples)}")

Train set: 80 tickets
Test set: 20 tickets
Few-shot examples: 4


## 3. Zero-Shot Classification

In [ ]:
# Initialize Azure OpenAI (GPT-4.1) for classification
print("Initializing Azure OpenAI GPT-4.1 for classification...")

llm = AzureChatOpenAI(
    deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_CHAT"),
    model_name="gpt-4.1",
    openai_api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    openai_api_base=os.getenv("AZURE_OPENAI_ENDPOINT"),
    openai_api_type="azure",
    openai_api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0.3
)

categories = ["Billing", "Technical", "Account", "General"]
print("✓ GPT-4.1 Model loaded!")

# Create classification prompt template
classification_prompt = PromptTemplate(
    input_variables=["ticket"],
    template="""Classify the following support ticket into ONE of these categories: Billing, Technical, Account, or General.

Ticket: {ticket}

Return ONLY the category name, nothing else."""
)


WARNING! azure_deployment is not default parameter.
                    azure_deployment was transferred to model_kwargs.
                    Please confirm that azure_deployment is what you intended.
WARNING! azure_endpoint is not default parameter.
                    azure_endpoint was transferred to model_kwargs.
                    Please confirm that azure_endpoint is what you intended.
WARNING! api_type is not default parameter.
                    api_type was transferred to model_kwargs.
                    Please confirm that api_type is what you intended.


Initializing Azure OpenAI GPT-4.1 for classification...


ValidationError: 1 validation error for AzureChatOpenAI
__root__
  Did not find openai_api_base, please add an environment variable `OPENAI_API_BASE` which contains it, or pass  `openai_api_base` as a named parameter. (type=value_error)

In [ ]:
# Azure OpenAI (GPT-4.1) classification
print("Performing GPT-4.1 classification on test set...")
zero_shot_preds = []  # Using this variable for compatibility
zero_shot_scores = []

for i, ticket in enumerate(X_test):
    try:
        # Use LLM to classify
        prompt = classification_prompt.format(ticket=ticket)
        response = llm.predict(prompt)
        predicted_label = response.strip()
        
        # Validate prediction
        if predicted_label not in categories:
            # If invalid, find closest match
            predicted_label = categories[0]
        
        zero_shot_preds.append(predicted_label)
        # Use confidence score (all 1.0 from LLM)
        scores = [1.0 if cat == predicted_label else 0.0 for cat in categories]
        zero_shot_scores.append(scores)
        
        if (i + 1) % 5 == 0:
            print(f"  Processed {i+1}/{len(X_test)} tickets")
    except Exception as e:
        print(f"  Error on ticket {i}: {e}")
        zero_shot_preds.append(categories[0])

print(f"\n✓ GPT-4.1 classification completed!")
print(f"Sample predictions:")
for i in range(min(5, len(X_test))):
    print(f"  Ticket: {X_test.iloc[i][:60]}...")
    print(f"  Predicted: {zero_shot_preds[i]}, True: {y_test.iloc[i]}")
    print()


In [ ]:
# Zero-shot evaluation
zero_shot_accuracy = accuracy_score(y_test, zero_shot_preds)
zero_shot_f1 = f1_score(y_test, zero_shot_preds, average='weighted')

print("=== Zero-Shot Results ===")
print(f"Accuracy: {zero_shot_accuracy:.4f}")
print(f"F1-Score (Weighted): {zero_shot_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, zero_shot_preds, target_names=categories))

## 4. Few-Shot Learning Classification

In [ ]:
# Few-shot learning with GPT-4.1
# Use examples in the prompt for better performance
few_shot_template = PromptTemplate(
    input_variables=["ticket"],
    template="""Classify support tickets into: Billing, Technical, Account, or General.

Examples:
Ticket: "I was charged twice for my subscription this month. Can you refund the duplicate charge?"
Category: Billing

Ticket: "The app keeps crashing when I try to upload files. Error code 500."
Category: Technical

Ticket: "I forgot my password. How can I reset it?"
Category: Account

Ticket: "What are your business hours? When can I call support?"
Category: General

Now classify this ticket:
Ticket: {ticket}

Return ONLY the category name."""
)

print("Performing few-shot GPT-4.1 classification on test set...")
few_shot_preds = []
few_shot_scores = []

for i, ticket in enumerate(X_test):
    try:
        prompt = few_shot_template.format(ticket=ticket)
        response = llm.predict(prompt)
        predicted_label = response.strip()
        
        if predicted_label not in categories:
            predicted_label = categories[0]
        
        few_shot_preds.append(predicted_label)
        scores = [1.0 if cat == predicted_label else 0.0 for cat in categories]
        few_shot_scores.append(scores)
        
        if (i + 1) % 5 == 0:
            print(f"  Processed {i+1}/{len(X_test)} tickets")
    except Exception as e:
        print(f"  Error on ticket {i}: {e}")
        few_shot_preds.append(categories[0])

print(f"\n✓ Few-shot GPT-4.1 classification completed!")


In [ ]:
# Few-shot evaluation
few_shot_accuracy = accuracy_score(y_test, few_shot_preds)
few_shot_f1 = f1_score(y_test, few_shot_preds, average='weighted')

print("=== Few-Shot Results ===")
print(f"Accuracy: {few_shot_accuracy:.4f}")
print(f"F1-Score (Weighted): {few_shot_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, few_shot_preds, target_names=categories))

## 5. Top-3 Tag Prediction

In [ ]:
# Extract top-3 predictions
top3_predictions = []

for i, scores in enumerate(zero_shot_scores):
    # Get top 3 indices
    top3_indices = np.argsort(scores)[::-1][:3]
    top3_tags = [(categories[idx], scores[idx]) for idx in top3_indices]
    top3_predictions.append({
        'ticket': X_test.iloc[i],
        'true_label': y_test.iloc[i],
        'top3_predictions': top3_tags
    })

print("=== Top-3 Predictions (First 5 Examples) ===")
for i, pred in enumerate(top3_predictions[:5]):
    print(f"\n{i+1}. Ticket: {pred['ticket'][:60]}...")
    print(f"   True Label: {pred['true_label']}")
    print(f"   Top-3 Predictions:")
    for rank, (tag, score) in enumerate(pred['top3_predictions'], 1):
        print(f"     {rank}. {tag}: {score:.3f}")

In [ ]:
# Calculate top-3 accuracy (prediction is correct if true label is in top 3)
top3_accuracy = sum([
       pred['true_label'] in [tag for tag, _ in pred['top3_predictions']]
    for pred in top3_predictions
]) / len(top3_predictions)

print(f"\n=== Top-3 Accuracy ===")
print(f"Top-3 Accuracy: {top3_accuracy:.4f} ({100*top3_accuracy:.2f}%)")

## 6. Evaluation & Visualization

In [ ]:
# Compare approaches
comparison_results = pd.DataFrame({
    'Approach': ['Zero-Shot', 'Few-Shot', 'Top-3 (Zero-Shot)'],
    'Accuracy': [zero_shot_accuracy, few_shot_accuracy, top3_accuracy],
    'F1-Score': [
        f1_score(y_test, zero_shot_preds, average='weighted'),
        f1_score(y_test, few_shot_preds, average='weighted'),
        top3_accuracy
    ]
})

print("\n=== Approach Comparison ===")
print(comparison_results)

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison_results))
width = 0.35

ax.bar(x - width/2, comparison_results['Accuracy'], width, label='Accuracy', color='skyblue', edgecolor='navy')
ax.bar(x + width/2, comparison_results['F1-Score'], width, label='F1-Score', color='lightcoral', edgecolor='darkred')

ax.set_xlabel('Approach', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Classification Approach Comparison', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(comparison_results['Approach'])
ax.legend(fontsize=11)
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('approach_comparison.png', dpi=300)
plt.show()
print("\nComparison plot saved!")

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Zero-shot confusion matrix
cm_zero_shot = confusion_matrix(y_test, zero_shot_preds, labels=categories)
sns.heatmap(cm_zero_shot, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=categories, yticklabels=categories, cbar_kws={'label': 'Count'})
axes[0].set_ylabel('True Label', fontsize=11)
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_title('Confusion Matrix - Zero-Shot', fontsize=12)

# Few-shot confusion matrix
cm_few_shot = confusion_matrix(y_test, few_shot_preds, labels=categories)
sns.heatmap(cm_few_shot, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=categories, yticklabels=categories, cbar_kws={'label': 'Count'})
axes[1].set_ylabel('True Label', fontsize=11)
axes[1].set_xlabel('Predicted Label', fontsize=11)
axes[1].set_title('Confusion Matrix - Few-Shot', fontsize=12)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300)
plt.show()
print("Confusion matrices saved!")

In [ ]:
# Per-class performance
zero_shot_precision = precision_score(y_test, zero_shot_preds, average=None, labels=categories)
zero_shot_recall = recall_score(y_test, zero_shot_preds, average=None, labels=categories)
zero_shot_f1_per_class = f1_score(y_test, zero_shot_preds, average=None, labels=categories)

perfomance_df = pd.DataFrame({
    'Category': categories,
    'Precision': zero_shot_precision,
    'Recall': zero_shot_recall,
    'F1-Score': zero_shot_f1_per_class
})

print("\n=== Per-Class Performance (Zero-Shot) ===")
print(performace_df)

# Visualize per-class performance
plt.figure(figsize=(12, 5))
x = np.arange(len(categories))
width = 0.25

plt.bar(x - width, zero_shot_precision, width, label='Precision', color='skyblue', edgecolor='navy')
plt.bar(x, zero_shot_recall, width, label='Recall', color='lightcoral', edgecolor='darkred')
plt.bar(x + width, zero_shot_f1_per_class, width, label='F1-Score', color='lightgreen', edgecolor='darkgreen')

plt.xlabel('Category', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Zero-Shot Performance Per Category', fontsize=14)
plt.xticks(x, categories)
plt.legend(fontsize=11)
plt.ylim([0, 1])
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('per_class_performance.png', dpi=300)
plt.show()
print("Per-class performance plot saved!")

## 7. Summary & Insights

In [ ]:
# Save top-3 predictions
with open('top3_predictions.json', 'w') as f:
    json.dump([
        {
            'ticket': pred['ticket'],
            'true_label': pred['true_label'],
            'top3_predictions': [(tag, float(score)) for tag, score in pred['top3_predictions']]
        }
        for pred in top3_predictions
    ], f, indent=2)

print("Top-3 predictions saved to 'top3_predictions.json'")

In [ ]:
# Summary
summary = f"""
=== TASK 5: AUTO TAGGING SUPPORT TICKETS - SUMMARY ===

DATASET:
- Total tickets: {len(df)}
- Training set: {len(X_train)} tickets
- Test set: {len(X_test)} tickets
- Categories: {', '.join(categories)}
- Class balance: Balanced (25% each)

APPROACHES EVALUATED:

1. ZERO-SHOT CLASSIFICATION
   - Model: facebook/bart-large-mnli
   - Accuracy: {zero_shot_accuracy:.4f} ({100*zero_shot_accuracy:.2f}%)
   - F1-Score: {f1_score(y_test, zero_shot_preds, average='weighted'):.4f}
   - Pros: No training required, instant deployment
   - Cons: May not capture domain-specific patterns

2. FEW-SHOT LEARNING
   - Model: facebook/bart-large-mnli with context examples
   - Accuracy: {few_shot_accuracy:.4f} ({100*few_shot_accuracy:.2f}%)
   - F1-Score: {f1_score(y_test, few_shot_preds, average='weighted'):.4f}
   - Pros: Better context awareness, example-driven
   - Cons: Requires curating examples

3. TOP-3 PREDICTION
   - Top-3 Accuracy: {top3_accuracy:.4f} ({100*top3_accuracy:.2f}%)
   - Pros: Provides ranking of alternatives
   - Cons: May surface multiple equally likely tags

PERFORMANCE COMPARISON:
- Best Single-Label Accuracy: {max(zero_shot_accuracy, few_shot_accuracy):.4f}
- Top-3 Coverage: {top3_accuracy:.4f}
- Improvement (Few-Shot vs Zero-Shot): {(few_shot_accuracy - zero_shot_accuracy):.4f}

PER-CLASS METRICS (Zero-Shot):
"""

for cat, prec, rec, f1 in zip(categories, zero_shot_precision, zero_shot_recall, zero_shot_f1_per_class):
    summary += f"  - {cat}: Precision={prec:.3f}, Recall={rec:.3f}, F1={f1:.3f}\n"

summary += f"""

KEY INSIGHTS:
- Zero-shot classification provides reasonable baseline accuracy
- Few-shot learning can improve performance with domain examples
- Top-3 predictions offer confidence in ambiguous cases
- Zero-shot approach is practical for immediate deployment
- Could be further improved with fine-tuning on domain data

RECOMMENDATIONS:
1. Start with zero-shot for quick implementation
2. Use few-shot examples for known problematic categories
3. Consider fine-tuning if dataset grows beyond 1000 samples
4. Implement human review for low-confidence predictions
5. Monitor and update model as support patterns evolve
"""

print(summary)

# Save summary
with open('results_summary.txt', 'w') as f:
    f.write(summary)
print("\nSummary saved to 'results_summary.txt'")